# BUTQDB: download and load the data

ECG quality dataset from PhysioNet: 18 recordings of about 24 hours each from 15 people, wearing a chest ECG sensor (Bittium Faros 180).
Experts marked every part of each recording with a quality class: **1** = clean, **2** = noisy but heartbeats visible, **3** = unusable.

Run this notebook from the **repo root**. The data goes into `data/`, which is in `.gitignore` and never pushed to GitHub.

## 1. Download the dataset

Downloads all files (about 4.2 GB) from PhysioNet's official mirror into `data/`.
Files that are already there are skipped, so you can run the cell again if it stops halfway.

In [ ]:
import urllib.request
from pathlib import Path

BASE = "https://physionet-open.s3.amazonaws.com/butqdb/1.0.0/"   # PhysioNet's official mirror
DATA = Path("data")

def get(path):
    dest = DATA / path
    if dest.exists():
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    print("downloading", path)
    tmp = dest.with_name(dest.name + ".part")
    urllib.request.urlretrieve(BASE + path, tmp)
    tmp.replace(dest)

get("RECORDS")
get("subject-info.csv")
records = sorted({line.split("/")[0] for line in (DATA / "RECORDS").read_text().split()})

for rec in records:
    for f in ["ECG.hea", "ECG.dat", "ACC.hea", "ACC.dat", "ANN.csv"]:
        get(f"{rec}/{rec}_{f}")

print("done:", len(records), "records")

downloading RECORDS
downloading subject-info.csv
downloading 100001/100001_ECG.hea
downloading 100001/100001_ECG.dat
downloading 100001/100001_ACC.hea
downloading 100001/100001_ACC.dat
downloading 100001/100001_ANN.csv
downloading 100002/100002_ECG.hea
downloading 100002/100002_ECG.dat
downloading 100002/100002_ACC.hea
downloading 100002/100002_ACC.dat
downloading 100002/100002_ANN.csv
downloading 103001/103001_ECG.hea
downloading 103001/103001_ECG.dat
downloading 103001/103001_ACC.hea
downloading 103001/103001_ACC.dat
downloading 103001/103001_ANN.csv
downloading 103002/103002_ECG.hea
downloading 103002/103002_ECG.dat
downloading 103002/103002_ACC.hea
downloading 103002/103002_ACC.dat
downloading 103002/103002_ANN.csv
downloading 103003/103003_ECG.hea
downloading 103003/103003_ECG.dat
downloading 103003/103003_ACC.hea
downloading 103003/103003_ACC.dat
downloading 103003/103003_ANN.csv
downloading 104001/104001_ECG.hea
downloading 104001/104001_ECG.dat
downloading 104001/104001_ACC.hea

## 2. Function to load one recording

Each recording is 24 hours long, so we load only a time slice (default: the first 10 minutes).

- **ECG**: heart signal, 1000 samples per second, in µV
- **ACC**: movement on 3 axes (x, y, z), 100 samples per second, in mg
- **Annotations**: the experts' quality labels as `start`, `end` (sample numbers, counting from 1) and `quality` (0 = not annotated)

In [ ]:
import wfdb
import pandas as pd

def load_record(rec, start_min=0, minutes=10):
    a = int(start_min * 60 * 1000)          # ECG is 1000 Hz
    b = a + int(minutes * 60 * 1000)
    ecg = wfdb.rdrecord(f"data/{rec}/{rec}_ECG", sampfrom=a, sampto=b)
    acc = wfdb.rdrecord(f"data/{rec}/{rec}_ACC", sampfrom=a // 10, sampto=b // 10)   # ACC is 100 Hz
    ann = pd.read_csv(f"data/{rec}/{rec}_ANN.csv", header=None).iloc[:, 9:12].dropna().astype(int)
    ann.columns = ["start", "end", "quality"]   # consensus labels, sample numbers start at 1
    return ecg, acc, ann

subjects = pd.read_csv("data/subject-info.csv", sep=";")

## 3. Check that every recording loads

Loads 1 minute from each recording and prints the sampling rates, the units and how many annotated segments each quality class has.
The table below lists the participants (sex, age, height, weight, smoker).
Note: the first 3 digits of the ID are the person, so `100001` and `100002` are the same person.

In [ ]:
for rec in records:
    ecg, acc, ann = load_record(rec, minutes=1)
    print(rec, ecg.fs, ecg.units, acc.fs, acc.units, ann["quality"].value_counts().to_dict())

subjects